# TSCD: Financial Regulation Classification Dataset

Este notebook procesa las simulaciones financieras (outputdata) para generar un dataset de grafos (PyTorch Geometric).
El objetivo es clasificar el tipo de regulación (SRT, TOBIN, NINGUNO) basándose en la topología promediada.

## 1. Creación del dataset e ingeniería de características topológicas.

Este cuaderno tiene como objetivo transformar los resultados brutos de las simulaciones del modelo basado en agentes (ABM) en un dataset estructurado compatible con PyTorch Geometric. El proceso consiste en ingerir los archivos parquet generados por cada ejecución, colapsar la dimensión temporal mediante promedios y enriquecer la información financiera con métricas topológicas derivadas de la estructura de la red interbancaria.

El resultado final será una colección de objetos Data (grafos), donde cada nodo (banco) tendrá un vector de características híbrido: tres variables de estado financiero (liquidez, equity, debtrank) y tres métricas de centralidad (clustering, betweenness, eigenvector). Estas estructuras se etiquetarán según el escenario regulatorio (NO_TAX, TOBIN, SRT) y se guardarán en un archivo .pt listo para el entrenamiento de modelos de aprendizaje profundo geométrico.

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import networkx as nx
import torch
from torch_geometric.data import Data
from tqdm import tqdm

In [ ]:
# Definición de rutas y mapeo de etiquetas
DATA_DIR = "outputdata"
OUTPUT_FILE = "dataset_interbancario.pt"

# Mapeo de clases a enteros para el aprendizaje supervisado
LABEL_MAP = {
    "NINGUNO": 0,  # Se renombrará conceptualmente a NO_TAX
    "TOBIN": 1,
    "SRT": 2
}

print(f"Librerías cargadas. Leyendo datos desde: {DATA_DIR}")

Comenzamos definiendo la lógica de procesamiento para una única simulación. La función principal se encargará de reconstruir el "grafo promedio" de todo el periodo simulado. Para ello, leemos los archivos de bancos y de la red interbancaria sin realizar ningún filtrado temporal, capturando así la dinámica completa desde el inicio hasta el estado estacionario.Primero, agrupamos los datos de los bancos por su identificador para obtener las medias de liquidez, patrimonio y DebtRank. Paralelamente, condensamos la red interbancaria promediando los pesos de las transacciones entre cada par de bancos (origen-destino). Con esta lista de aristas promediada, instanciamos un grafo dirigido en NetworkX, asegurándonos de añadir explícitamente todos los nodos presentes en el dataframe de bancos para no excluir aquellos que pudieran haber quedado aislados (sin conexiones).Una vez construido el grafo, calculamos las métricas topológicas. El coeficiente de clustering, la intermediación (betweenness) y la centralidad de vector propio (eigenvector) se computan para cada nodo y se fusionan con las métricas financieras previamente calculadas. Este paso nos genera una matriz de características completa de dimensiones $(N_{bancos} \times 6)$. Finalmente, convertimos estas estructuras a tensores de PyTorch y empaquetamos todo en un objeto Data, asignándole la etiqueta correspondiente al tipo de impuesto aplicado en esa simulación.

In [ ]:
def procesar_simulacion(run_path, label_code):
    """
    Procesa una carpeta de simulación y devuelve un objeto PyTorch Geometric Data.
    """
    try:
        # 1. Carga de datos crudos (sin filtrar tiempo)
        path_banks = os.path.join(run_path, "banks.parquet")
        path_net = os.path.join(run_path, "net_BB.parquet")
        
        if not (os.path.exists(path_banks) and os.path.exists(path_net)):
            return None
            
        df_banks = pd.read_parquet(path_banks)
        df_net = pd.read_parquet(path_net)
        
        # 2. Agregación Temporal (Promedios Globales)
        # Bancos: Media de estados financieros por ID
        # Asumimos columnas: id, liq, eq, dr, t
        bank_features = df_banks.groupby("id")[["liq", "eq", "dr"]].mean().sort_index()
        num_nodes = len(bank_features)
        
        # Red: Media de pesos de aristas
        edge_means = df_net.groupby(["source", "target"])["weight"].mean().reset_index()
        
        # 3. Construcción del Grafo con NetworkX para métricas
        G = nx.DiGraph()
        G.add_nodes_from(bank_features.index) # Añadir todos los nodos (incluso aislados)
        
        # Añadir aristas con peso promedio
        for _, row in edge_means.iterrows():
            G.add_edge(row["source"], row["target"], weight=row["weight"])
            
        # 4. Cálculo de Métricas Topológicas
        # Clustering (para grafos dirigidos se usa la versión estándar o se convierte, aquí usamos la implementación de nx)
        clustering = nx.clustering(G, weight="weight")
        
        # Betweenness Centrality (usando pesos si se desea, o puramente estructural)
        betweenness = nx.betweenness_centrality(G, weight="weight")
        
        # Eigenvector Centrality (manejo de excepciones por convergencia o grafos desconectados)
        try:
            eigenvector = nx.eigenvector_centrality(G, weight="weight", max_iter=1000)
        except:
            # Fallback seguro: centralidad plana o ceros si no converge
            eigenvector = {node: 0.0 for node in G.nodes()}
            
        # 5. Ensamblaje del Vector de Características (Feature Matrix)
        # Ordenamos las métricas por ID de nodo para alinear con bank_features
        topo_df = pd.DataFrame({
            "clustering": pd.Series(clustering),
            "betweenness": pd.Series(betweenness),
            "eigenvector": pd.Series(eigenvector)
        }).sort_index()
        
        # Concatenación final: [liq, eq, dr, clust, betw, eigen]
        full_features = pd.concat([bank_features, topo_df], axis=1).fillna(0.0)
        
        # 6. Creación del Objeto PyTorch Geometric
        x = torch.tensor(full_features.values, dtype=torch.float)
        
        # Edge Index y Edge Attr
        # Requerimos formato COO (2, num_edges)
        sources = edge_means["source"].values
        targets = edge_means["target"].values
        weights = edge_means["weight"].values
        
        edge_index = torch.tensor([sources, targets], dtype=torch.long)
        edge_attr = torch.tensor(weights, dtype=torch.float).view(-1, 1)
        
        y = torch.tensor([label_code], dtype=torch.long)
        
        data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y)
        
        return data

    except Exception as e:
        print(f"Error procesando {run_path}: {e}")
        return None

Con la lógica encapsulada, procedemos a iterar sobre el sistema de archivos. Buscamos todas las carpetas dentro del directorio de datos que coincidan con el patrón de nomenclatura de nuestras simulaciones. Para cada carpeta encontrada, extraemos el tipo de simulación del nombre del directorio y le asignamos su etiqueta numérica correspondiente utilizando el mapa definido al inicio. Nótese que las carpetas etiquetadas como "NINGUNO" se procesan bajo la etiqueta clase 0 (equivalente a NO_TAX).

Utilizamos una barra de progreso para monitorizar el avance, ya que el cálculo de centralidades (especialmente eigenvector y betweenness) puede ser computacionalmente intensivo si la red es densa o grande. Los grafos resultantes se acumulan en una lista en memoria.

In [ ]:
dataset_list = []
sim_folders = glob.glob(os.path.join(DATA_DIR, "run_*"))

print(f"Se encontraron {len(sim_folders)} carpetas de simulación.")

for folder in tqdm(sim_folders, desc="Generando Grafos"):
    folder_name = os.path.basename(folder)
    
    # Extraer tipo (SRT, TOBIN, NINGUNO)
    # Formato esperado: run_TIPO_sim_NUM
    parts = folder_name.split("_")
    if len(parts) >= 3:
        sim_type = parts[1]
        
        if sim_type in LABEL_MAP:
            label = LABEL_MAP[sim_type]
            
            data_obj = procesar_simulacion(folder, label)
            
            if data_obj is not None:
                dataset_list.append(data_obj)

print(f"\nProcesamiento completado. Total de grafos generados: {len(dataset_list)}")

Para finalizar, guardamos la lista completa de objetos geométricos en el archivo .pt especificado. Este archivo contendrá el dataset crudo, preservando la distribución original de los datos para permitir una división adecuada entre entrenamiento y prueba (train/test split) en etapas posteriores, antes de aplicar cualquier técnica de estandarización o normalización. Imprimimos una muestra del primer elemento del dataset para verificar visualmente que las dimensiones de la matriz de características (x), el índice de aristas (edge_index) y la etiqueta (y) son coherentes con lo esperado (N nodos x 6 features).

In [ ]:
# Guardar el dataset en disco
torch.save(dataset_list, OUTPUT_FILE)
print(f"Dataset guardado exitosamente en: {OUTPUT_FILE}")

# Inspección rápida del primer elemento
if dataset_list:
    sample = dataset_list[0]
    print("\n--- Inspección de Muestra (Grafo 0) ---")
    print(f"Estructura: {sample}")
    print(f"Dimensiones de Features (x): {sample.x.shape}  [Esperado: (N, 6)]")
    print(f"Etiqueta (y): {sample.y.item()}  [Mapeo: {LABEL_MAP}]")
    print(f"Número de aristas: {sample.num_edges}")
    print("Features (primeras 2 filas):")
    print(sample.x[:2])
else:
    print("Advertencia: La lista de datos está vacía.")